# Estimation: Kriging, Inverse Distance Weighting (IDW) and K-Nearest Neighbors (KNN)

This notebook demonstrates how to perform estimation using Seequent Evo's Geostatistical Compute tasks.

Before running this notebook, create a Seequent Evo workspace and populate it with the example data using [1-populate-workspace.ipynb](./1-populate-workspace.ipynb).

---

First, import a few python packages.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import evo.compute.tasks as evo_tasks
import evo.compute.tasks.geostatistics.idw as idw
import evo.compute.tasks.geostatistics.knn as knn
import evo.compute.tasks.geostatistics.kriging as kriging
import evo.objects.typed as evo_objs
from evo.notebooks import ServiceManagerWidget

pio.templates.default = "simple_white"
%load_ext evo.widgets

## Authenticate
Log into Seequent Evo and select the appropriate workspace. It is assumed that prior to running this example notebook a Seequent Evo workspace has been created and populated with the necessary example data. If not, refer to the prior notebook [1-populate-workspace.ipynb`](./1-populate-workspace.ipynb)

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(client_id="<CLIENT-ID>").login()

## Load the Objects Needed

Note that when accessing an object from the Evo workspace, a reference to that object is returned.

This reference contains information about the unique ID and location of that object as well as the coordinates and attributes. What this object reference does **not** contain is the actual data. 

The object reference and the tabular data of the object are separated such that workflows can be orchestrated on Evo without ever having to download the data locally. However, if access to the underlying data is desired, use the `.to_dataframe` method as needed to access the numerical data associated with the object.

In this case, a pointset and the blockmodel are needed.

In [ ]:
comps = await evo_objs.object_from_path(evo_connection, "WP comps.json")
variogram = await evo_objs.object_from_path(evo_connection, "Cu_pct Variogram Model.json")
block_model = await evo_objs.object_from_path(evo_connection, "WP Regular BlockModel.json")

Each object, such as the variogram, can be inspected via its html representation, which includes linkes to the `Portal` and the `Viewer`.

In [ ]:
variogram

To inspect the variogram in each principal direction, leverage a python based potting package. Extract the information about each principal direction from the variogram model and plot. In this case [plotly](https://docs.plotly.com/) is used.

In [ ]:
directions = variogram.get_principal_directions()

fig = px.line(
    {
        "distance": np.concatenate([curve.distance for curve in directions]),
        "semivariance": np.concatenate([curve.semivariance for curve in directions]),
        "direction": np.concatenate([np.repeat(curve.direction, len(curve.distance)) for curve in directions]),
    },
    x="distance",
    y="semivariance",
    facet_col="direction",
    labels={"distance": "Distance", "semivariance": "Semivariance"},
    title="Variogram by principal direction",
)
fig.show()

## Generate the Estimation Parameters

First, create the search ellipsoid and neighborhood that can be used for multiple estimations. The ellipsoid (called `ell` in this example) is generated from the variogram ranges and orientations, then scaled to be 1.5x the size of the variogram ellipsoid.

### Kriging

In [ ]:
ell = variogram.get_ellipsoid().scaled(1.5)
srch = evo_tasks.SearchNeighborhood(ellipsoid=ell, max_samples=30, min_samples=1)

In [ ]:
krig_params = kriging.KrigingParameters(
    source=comps.attributes["Cu_pct"],
    target=block_model.attributes["Cu_OK"],
    variogram=variogram,
    neighborhood=srch,
    block_discretisation=kriging.BlockDiscretisation(nx=5, ny=5, nz=2),
    diagnostics=kriging.KrigingDiagnostics(kriging_variance=True),
)

#### Optional: request kriging diagnostics

Kriging can write per-location **diagnostics** onto the target object alongside the estimate. Only the diagnostics you ask for are computed, and each one gets its own attribute — so every name must differ from the estimate's attribute name and from the other diagnostics.

Each field of `KrigingDiagnostics` accepts:

- `True` — create an attribute using the recommended name (these match Leapfrog's conventions, so results are familiar once imported).
- a string — create an attribute with that name instead.
- an attribute from the target object — update it if it already exists, otherwise create it.

```python
from evo.compute.tasks import KrigingDiagnostics

diagnostics = KrigingDiagnostics(
    kriging_variance=True,  # creates "KV"
    kriging_efficiency="CU_efficiency",  # creates "CU_efficiency"
    num_samples=block_model.attributes["NS"],  # updates "NS" if it already exists
)
```

| Diagnostic | Recommended name | Description |
| --- | --- | --- |
| `valid` | `valid` | Whether the location's neighbourhood satisfied the search constraints and produced an estimate. |
| `kriging_variance` | `KV` | Kriging variance (estimation uncertainty). |
| `slope_of_regression` | `SoR` | Slope of regression, a diagnostic for conditional bias. |
| `kriging_efficiency` | `KE` | Proportion of point or intra-block variance explained by the kriging weights. |
| `kriging_mean` | `KM` | GLS mean for ordinary kriging, or the supplied constant mean for simple kriging. |
| `num_samples` | `NS` | Number of data samples used in the estimate. |
| `num_drillholes` | `NDh` | Number of distinct drillholes contributing to the estimate. Requires a downhole-intervals source. |
| `num_duplicates` | `ND` | Number of duplicate sample locations used in the estimate. |
| `num_equidistant` | `NeD` | Hint of how many other samples could have replaced the last returned sample because they were the same distance away. |
| `sum_weights` | `Sum` | Sum of the kriging weights applied to data samples. |
| `sum_positive_weights` | `SumP` | Sum of the positive kriging weights. |
| `sum_negative_weights` | `SumN` | Sum of the negative kriging weights. |
| `min_distance` | `MinD` | Minimum isotropic Euclidean distance from the location to any neighbour. |
| `mean_distance` | `AvgD` | Mean isotropic Euclidean distance from the location to all neighbours. |
| `aniso_min_distance` | `MinAD` | Minimum anisotropic (ellipsoid-space) distance from the location to any neighbour. |
| `aniso_mean_distance` | `AvgAD` | Mean anisotropic (ellipsoid-space) distance from the location to all neighbours. |


### Inverse Distance Weighting (IDW)

In [ ]:
idw_params = idw.IDWParameters(
    source=comps.attributes["Cu_pct"], target=block_model.attributes["Cu_IDW"], neighborhood=srch, power=2
)

### K-Nearest Neighbors (KNN)

In [ ]:
knn_params = knn.KNNParameters(
    source=comps.attributes["Cu_pct"], target=block_model.attributes["Cu_KNN"], neighborhood=srch, k=1
)

## Execute the Estimation Tasks

Now that parameters for all tasks have been defined, they can be executed concurrently. Again note the `await` used here to execute tasks asyncronously meaning there is no seequence or waiting for each task to finish, they all run simultaneosly and will communicate their status back to the user. As the tasks conclude their computational work, results will be saved to the block model and available on blocksync or can be imported into leapfrog immediately with by reloading the blocksync model.

In [ ]:
results = await evo_tasks.run(evo_connection, [krig_params, idw_params, knn_params], preview=True)

In [ ]:
results

In [ ]:
updated_block_model = await block_model.refresh()
updated_block_model

In [ ]:
block_model_df = await updated_block_model.to_dataframe(columns=["Cu_KNN", "Cu_IDW", "Cu_OK"])
block_model_df.head()

In [ ]:
comps_df = await comps.to_dataframe()
comps_df.head()

In [ ]:
composites_df = comps_df[["Cu_pct"]].rename(columns={"Cu_pct": "Cu (%)"})
composites_df["Method"] = "Composites"

estimates_df = block_model_df[["Cu_KNN", "Cu_IDW", "Cu_OK"]].melt(
    var_name="Method",
    value_name="Cu (%)",
)
histogram_df = pd.concat([composites_df, estimates_df], ignore_index=True)
histogram_df.head()

In [ ]:


fig = px.histogram(
    histogram_df,
    x="Cu (%)",
    color="Method",
    category_orders={"Method": ["Composites", "Cu_KNN", "Cu_IDW", "Cu_OK"]},
    color_discrete_sequence=["#071E22", "#1D7874", "#F4C095", "#EE2E31"],
    barmode="overlay",
    opacity=0.8,
    nbins=50,
    histnorm="probability density",
    title="Composites vs Estimation Results",
    labels={"Cu (%)": "Cu (%)"},
)
fig.update_traces(bingroup="cu_comparison")
fig.show()

In [ ]:
histogram_df.groupby("Method").describe()